# Storing Chat History in 3rd Party Storage

### By default, when using `ChatAgent`, chat history is stored in memory in the `AgentThread` object or the underlying inference service, if the service supports it
### But we need persistence chat history, for which the external database is to be used.
### Here we will use Redis database to store the chat history, the in-memory behavior will not be possible if you want to buid any real production agents

## First Step is to install Redis

In [2]:
# ensure you have docker installed
# Run Redis official image using docker
!docker run -d --name my-redis -p 6379:6379 redis:latest

987b0dbc90f1f5a25a88134de4f6f426d91bc76311956353a6db529673d9210c


In [3]:
# check if the container is Running
!docker ps
# also you I can connect to the container to test it
# docker exec -it my-redis redis-cli

CONTAINER ID   IMAGE          COMMAND                  CREATED        STATUS                  PORTS                                         NAMES
987b0dbc90f1   redis:latest   "docker-entrypoint.s…"   1 second ago   Up Less than a second   0.0.0.0:6379->6379/tcp, [::]:6379->6379/tcp   my-redis


## Import Dependencies

Import the required libraries and Microsoft Agent Framework components:

- `asyncio`: For async/await support
- `os`: For accessing environment variables
- `json`: For JSON parsing
- `dotenv`: For loading environment variables from `.env` file
- `ChatAgent`: The main Agent class for building conversational AI agents
- `OpenAIChatClient`: Client for LLM inference using OpenAI-compatible endpoints (OpenRouter in this case)

In [1]:
# Import core dependencies to create the agent, for Agent Framework
import asyncio
import os
import json

from dotenv import load_dotenv, find_dotenv
# Core components for building Agent, tool-enabled agents
from agent_framework import ChatAgent
from agent_framework.openai import OpenAIChatClient
from agent_framework import AgentThread
from agent_framework.redis import RedisChatMessageStore

## Load Environment Variables

Load environment variables from a `.env` file in the project directory. This file should contain:
- `OPENROUTER_ENDPOINT`: The OpenRouter API endpoint URL
- `OPENROUTER_API_KEY`: Your OpenRouter API key

In [2]:
# load environment file
load_dotenv(find_dotenv())

True

## Setup Chat Client

Configure the `OpenAIChatClient` to use OpenRouter API, which provides access to various LLM models including NVIDIA's Nemotron model. The client is configured with:

- `base_url`: The OpenRouter API endpoint
- `api_key`: Your API key for authentication
- `model_id`: The specific model to use (NVIDIA Nemotron 3 Nano 30B in this case)

In [13]:
# Setup OpenAIChatClient for LLM Inference - Here we will use OpenRouter API which is compatible with OpenAI and NVIDIA 30B model
# This client connects to the OpenRouter Models which are OpenAI-compatible endpoint
# Environment variables required
# OPENROUTER_ENDPOINT - 
# OPENROUTER_API_KEY
openai_chat_client = OpenAIChatClient(
    base_url=os.environ.get("OPENROUTER_ENDPOINT"),
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    model_id="openai/gpt-oss-20b:free"
)

In [14]:
AGENT_NAME = "FoodAgent"

AGENT_INSTRUCTIONS = """You are an expert AI Chef dedicated to helping users discover and prepare delicious meals. Keep it concise, short and effective.
"""

## Create the Food Agent

Create the first agent (`food_agent`) with:

- `name`: "FoodAgent"
- `chat_client`: The OpenAI chat client configured earlier
- `instructions`: The behavior instructions defined above
- `chat_message_store`: External Database Store for persistent message.

## Basic example of using Redis chat message store.

In [15]:
 # Create Redis store with auto-generated thread ID
redis_store = RedisChatMessageStore(
        redis_url="redis://localhost:6379",
        # thread_id will be auto-generated if not provided
)

In [16]:
print(f"Created store with thread ID: {redis_store.thread_id}")

Created store with thread ID: thread_03e900f5-686a-480f-92a5-7ca46401fab1


In [17]:
thread = AgentThread(message_store=redis_store)

In [18]:
# create agent
# Here we create the foodAgent
# create the agent remember we are not using any tools here, this is simple example
food_agent = ChatAgent(
    name = AGENT_NAME,
    chat_client=openai_chat_client,
    instructions=AGENT_INSTRUCTIONS,
)

In [19]:
#### Have a conversation 
#### Starting conversation

In [20]:
query1 = "Hello, My name is Urula and I love Masala Chai"
response = await food_agent.run(query1, thread=thread)
print(response.text)

Nice to meet you, Urula! 🫶 So glad you love **Masala Chai**—it’s such a soul-warming drink. Whether you’re sipping it at home or dreaming of a cozy café, I’d love to help you:  
- Try a **new spice twist** (like cardamom-cinnamon or ginger-rose),  
- Make the **perfect creamy version** (with milk alternatives!),  
- Or even **pair it with snacks** (samosas? cookies?).  

Just say the word—I’ve got your chai cravings covered. 😊  
*What’s your favorite way to enjoy it?*


In [23]:
query2 = "I love spicier tea"
response2 = await food_agent.run(query2, thread=thread)
print(response2.text)

TypeError: TextReasoningContent.__init__() missing 1 required positional argument: 'text'